In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("gpu.csv", encoding="utf-8")

df = df[[
    "manufacturer",
    "base_clock_mhz",
    "boost_clock_mhz",
    "architecture",
    "foundry",
    "process_size_nm",
    "transistor_count_m",
    "transistor_density_k_mm2",
    "die_size_mm2",
    "bus_interface",
    "memory_clock_mhz",
    "memory_size_gb",
    "memory_bus_bits",
    "memory_bandwidth_gb_s",
    "memory_type",
    "shading_units",
    "texture_mapping_units",
    "render_output_processors",
    "streaming_multiprocessors",
    "tensor_cores",
    "ray_tracing_cores",
    "l1_cache_kb",
    "l2_cache_mb",
    "thermal_design_power_w",
    "release_date",
]]

df["year"] = pd.to_datetime(df["release_date"]).dt.year
df = df.drop(columns="release_date")
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

In [3]:
continuous_cols = [
    "base_clock_mhz",
    "boost_clock_mhz",
    "process_size_nm",
    "transistor_count_m",
    "transistor_density_k_mm2",
    "die_size_mm2",
    "bus_interface",
    "memory_clock_mhz",
    "memory_size_gb",
    "memory_bus_bits",
    "memory_bandwidth_gb_s",
    "memory_type",
    "shading_units",
    "texture_mapping_units",
    "render_output_processors",
    "streaming_multiprocessors",
    "tensor_cores",
    "ray_tracing_cores",
    "l1_cache_kb",
    "l2_cache_mb",
    "thermal_design_power_w",
]

categorical_cols = [
    "manufacturer",
    "architecture",
    "foundry",
    "year",
]

In [4]:
# Extract numbers from continuous columns
for col in continuous_cols:
    if df[col].dtype == "object":
        df[col] = df[col].str.extract(r"(\d+\.?\d*)").astype(float)

# Convert to numeric
df[continuous_cols] = df[continuous_cols].apply(pd.to_numeric, errors="coerce")

In [6]:
def fill_continuous_by_year_and_manufacturer(df, continuous_cols, group_cols=['year', 'manufacturer']):
    def recursive_fill_modes(modes_df, col):
        filled_modes = modes_df[col].copy()
        changed = True
        while changed:
            changed = False
            for idx in filled_modes[filled_modes.isna()].index:
                year, manufacturer = idx
                next_year = year + 1
                prev_year = year - 1
                min_year = filled_modes.index.get_level_values('year').min()
                max_year = filled_modes.index.get_level_values('year').max()

                if year == min_year:
                    next_idx = (next_year, manufacturer)
                    if next_idx in filled_modes.index and pd.notna(filled_modes[next_idx]):
                        filled_modes[idx] = filled_modes[next_idx] / 2
                        changed = True
                elif year == max_year:
                    prev_idx = (prev_year, manufacturer)
                    if prev_idx in filled_modes.index and pd.notna(filled_modes[prev_idx]):
                        filled_modes[idx] = filled_modes[prev_idx] * 2
                        changed = True
                else:
                    next_idx = (next_year, manufacturer)
                    prev_idx = (prev_year, manufacturer)
                    if next_idx in filled_modes.index and pd.notna(filled_modes[next_idx]):
                        filled_modes[idx] = filled_modes[next_idx] / 2
                        changed = True
                    elif prev_idx in filled_modes.index and pd.notna(filled_modes[prev_idx]):
                        filled_modes[idx] = filled_modes[prev_idx] * 2
                        changed = True
        return filled_modes

    # Step 1: Calculate yearly-manufacturer modes
    grouped_modes = df.groupby(group_cols)[continuous_cols].agg(
        lambda x: x.mode().iat[0] if not x.mode().empty else np.nan
    )

    # Ensure Int64 year index
    grouped_modes.index = grouped_modes.index.set_levels([
        grouped_modes.index.levels[0].astype("Int64"),
        grouped_modes.index.levels[1]
    ])

    # Step 2: Fill missing (year, manufacturer) combinations
    all_years = pd.Index(range(df["year"].min(), df["year"].max() + 1), dtype="Int64")
    all_manufacturers = df["manufacturer"].dropna().unique()
    full_index = pd.MultiIndex.from_product([all_years, all_manufacturers], names=['year', 'manufacturer'])
    grouped_modes = grouped_modes.reindex(full_index)

    # Step 3: Recursively fill missing modes
    for col in continuous_cols:
        grouped_modes[col] = recursive_fill_modes(grouped_modes, col)

    # Step 4: Apply filled values back to df
    for col in continuous_cols:
        df[col] = df.apply(
            lambda row: grouped_modes.loc[(row["year"], row["manufacturer"]), col]
            if pd.isna(row[col]) and pd.notna(row["year"]) and pd.notna(row["manufacturer"])
            else row[col],
            axis=1
        )

    return df


In [7]:
def fill_categorical_by_year_and_manufacturer(df, categorical_cols, group_cols=['year', 'manufacturer']):
    
    def calculate_modes_for_categorical(df, categorical_cols):
        grouped_modes = df.groupby(group_cols)[categorical_cols].agg(
            lambda x: x.mode().iat[0] if not x.mode().empty else np.nan
        )
        # Ensure proper index types (year as Int64)
        grouped_modes.index = grouped_modes.index.set_levels([
            grouped_modes.index.levels[0].astype("Int64"),
            grouped_modes.index.levels[1]
        ])
        return grouped_modes

    def recursive_fill_categorical_modes(modes_df, col):
        filled_modes = modes_df[col].copy()
        changed = True
        while changed:
            changed = False
            for idx in filled_modes[filled_modes.isna()].index:
                year, manufacturer = idx
                distance = 1

                while pd.isna(filled_modes[idx]) and (
                    (year - distance, manufacturer) in filled_modes.index or 
                    (year + distance, manufacturer) in filled_modes.index
                ):
                    prev_idx = (year - distance, manufacturer)
                    next_idx = (year + distance, manufacturer)

                    if prev_idx in filled_modes.index and pd.notna(filled_modes[prev_idx]):
                        filled_modes[idx] = filled_modes[prev_idx]
                        changed = True
                        break
                    if next_idx in filled_modes.index and pd.notna(filled_modes[next_idx]):
                        filled_modes[idx] = filled_modes[next_idx]
                        changed = True
                        break
                    distance += 1
        return filled_modes

    def add_missing_group_combinations(modes_df, all_years, all_manufacturers):
        full_index = pd.MultiIndex.from_product([all_years, all_manufacturers], names=['year', 'manufacturer'])
        return modes_df.reindex(full_index)

    # Step 1: Calculate grouped modes
    grouped_modes = calculate_modes_for_categorical(df, categorical_cols)

    # Step 2: Add missing combinations
    all_years = pd.Index(range(df["year"].min(), df["year"].max() + 1), dtype="Int64")
    all_manufacturers = df["manufacturer"].dropna().unique()
    grouped_modes = add_missing_group_combinations(grouped_modes, all_years, all_manufacturers)

    # Step 3: Recursively fill missing values
    for col in categorical_cols:
        grouped_modes[col] = recursive_fill_categorical_modes(grouped_modes, col)

    # Step 4: Apply filled values to df
    for col in categorical_cols:
        df[col] = df.apply(
            lambda row: grouped_modes.loc[(row["year"], row["manufacturer"]), col]
            if pd.isna(row[col]) and pd.notna(row["year"]) and pd.notna(row["manufacturer"])
            else row[col],
            axis=1
        )

    return df


In [8]:
fill_continuous_by_year_and_manufacturer(df, continuous_cols)
fill_categorical_by_year_and_manufacturer(df, categorical_cols)


,manufacturer,base_clock_mhz,boost_clock_mhz,architecture,foundry,process_size_nm,transistor_count_m,transistor_density_k_mm2,die_size_mm2,bus_interface,memory_clock_mhz,memory_size_gb,memory_bus_bits,memory_bandwidth_gb_s,memory_type,shading_units,texture_mapping_units,render_output_processors,streaming_multiprocessors,tensor_cores,ray_tracing_cores,l1_cache_kb,l2_cache_mb,thermal_design_power_w,year
0,ATI,10.0,10.0,Wonder,NEC,800.0,0.001953,0.021680,90.0,8.0,5.0,0.000032,32.0,0.020,0.000015,0,0,0,0,0,0,0.0,0.0,0.001404,1986
1,ATI,10.0,10.0,Wonder,NEC,800.0,0.001953,0.021680,90.0,8.0,5.0,0.000064,32.0,0.020,0.000015,0,0,0,0,0,0,0.0,0.0,0.001404,1986
2,ATI,25.0,25.0,Wonder,NEC,800.0,0.007812,0.086719,90.0,8.0,8.0,0.000250,32.0,0.032,0.000061,0,0,1,0,0,0,0.0,0.0,0.005615,1988
3,ATI,10.0,10.0,Wonder,NEC,800.0,0.003906,0.043359,90.0,8.0,5.0,0.000064,32.0,0.020,0.000031,0,0,0,0,0,0,0.0,0.0,0.002808,1987
4,ATI,10.0,10.0,Wonder,NEC,800.0,0.003906,0.043359,90.0,8.0,10.0,0.000250,32.0,0.040,0.000031,0,0,1,0,0,0,0.0,0.0,0.002808,1987
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2867,AMD,1660.0,2970.0,RDNA 4.0,TSMC,5.0,53900.000000,151000.000000,357.0,5.0,2518.0,16.000000,256.0,644.600,6.000000,4096,256,128,0,128,64,0.0,8.0,304.000000,2025
2868,Intel,300.0,2200.0,Xe2-LPG,TSMC,3.0,19600.000000,72100.000000,272.0,4.0,2375.0,0.000000,160.0,380.000,6.000000,896,56,28,0,112,7,0.0,4.0,35.000000,2025
2869,Intel,300.0,2350.0,Xe2-LPG,TSMC,3.0,19600.000000,72100.000000,272.0,4.0,2375.0,0.000000,160.0,380.000,6.000000,1024,64,32,0,128,8,0.0,4.0,35.000000,2025
2870,Intel,2500.0,2500.0,Xe2,TSMC,5.0,19600.000000,72100.000000,272.0,4.0,2375.0,10.000000,160.0,380.000,6.000000,2304,144,80,0,144,18,250.0,18.0,150.000000,2025


In [9]:
df = df.dropna(subset=['year'])

X = df.drop(columns=["year"])
y = df["year"]

categorical_cols = [col for col in categorical_cols if col != "year"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])  # Fit and transform on train
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

X_train = pd.get_dummies(X_train, columns=categorical_cols, dtype=int)
X_test = pd.get_dummies(X_test, columns=categorical_cols, dtype=int)

X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

In [ ]:
clf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

# Print the results
print(f"Mean Absolute Error (MAE): {mae:.4f} years")
print(f"Mean Squared Error (MSE): {mse:.4f}")

# Show prediction differences
comparison_df = pd.DataFrame({"Actual Year": y_test, "Predicted Year": y_pred})
comparison_df["Error"] = comparison_df["Predicted Year"] - comparison_df["Actual Year"]
print(comparison_df['Error'].value_counts().sort_index(), "\n")
bias = comparison_df['Error'].mean()
print(f"Average Prediction Bias: {bias:.2f} years")
print("")

feature_importance = pd.Series(clf.feature_importances_, index=X_train.columns)
print(feature_importance.sort_values(ascending=False))

Mean Absolute Error (MAE): 0.4794 years
Mean Squared Error (MSE): 0.5311
Error
-4.030000    1
-3.220000    1
-2.887500    1
-2.480000    1
-2.246667    1
            ..
 1.968333    1
 2.140000    1
 2.200000    1
 2.572500    1
 2.865000    1
Name: count, Length: 294, dtype: int64 

Average Prediction Bias: 0.08 years

process_size_nm                 0.818136
transistor_density_k_mm2        0.137847
memory_type                     0.007475
boost_clock_mhz                 0.005906
thermal_design_power_w          0.004535
                                  ...   
architecture_Kelvin             0.000000
architecture_RDNA 4.0           0.000000
architecture_Generation 5.75    0.000000
architecture_Xe-LPG             0.000000
foundry_SGS Microelettronica    0.000000
Length: 124, dtype: float64


In [12]:
print(comparison_df['Error'].describe(), "\n")

count     557.0
unique    294.0
top         0.0
freq       50.0
Name: Error, dtype: float64 



In [ ]:
# plt.figure(figsize=(2000, 50))  # Make the figure large enough to see all nodes
# plot_tree(
#     clf,                     # Your fitted decision tree
#     feature_names=X_train.columns.tolist(),   # Column names for features
#     filled=True,                   # Fill boxes with colors
#     rounded=True,                  # Rounded boxes
#     fontsize=10                    # Font size
# )
# plt.savefig("decision_tree.svg", format="svg", bbox_inches='tight')
# plt.show()